In [ ]:
import pandas as pd
import plotly.express as px
import torch
from darts.utils.statistics import plot_ccf, remove_seasonality, granger_causality_tests
from darts.utils.statistics import stationarity_test_adf
from darts.utils.utils import SeasonalityMode
from sklearn.preprocessing import StandardScaler

from aare.constants import TIME
from aare.params import read_params
from aare.preparation import resample, interpolate
from aare.remote_existenz_store import RemoteExistenzStore
from aare.utils import to_ts

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
# logging.basicConfig(level="DEBUG")

In [ ]:
params = read_params()
store = RemoteExistenzStore()

# Variables to look at

<https://api-datasette.konzept.space/existenz-api/hydro_parameters> \
<https://api-datasette.konzept.space/existenz-api/smn_parameters>

- Water temperature from Thun (hydro/temperature):
  - Thun is upstream from Bern, so how long do changes in the water temperature there take to come into effect in Bern? Are they negligible?
- Flow (hydro/flow):
  - Does the flow or change in flow have an influence on the temperature or change in temperature?
- Precipitation (smn/rr):
  - Does the precipitation have an influence on the temperature or change in temperature?
  - Does the precipitation have an influence on the flow? Probably yes, but at which lag is the correlation greatest?
  - To be useful as feature, pool together a total over some hours and make sure the model has access to all relevant lags.
- Sunshine duration (smn/ss):
  - Does the sunshine duration have an influence on the temperature or change in temperature?
  - How much lag is there in the heat transfer from the sunshine to the water temperature? could influence water directly without air temperature
- Air temperature (smn/tt):
  - How high is the correlation between air temperature and water temperature?
  - How much lag is there in the heat transfer from the air temperature to the water temperature?
  - Does the daily, weekly or monthly mean show correlation to the change in water temperature? might be nice feature
- Turbidity (hydro/turbidity):
  - Might influence the transfer rate from sunshine to water temperature
- Global Exposure (smn/rad):
  - Don't quite know what this is and does.
  - Probably just affects air temperature and won't make a good feature but idk.
- Relative Humidity (smn/rh):
  - Does humidity have any direct influence on the water temperature? If there is correlation, I suspect it's just because air temperature and precipitation affect RH.
  - Probably not a good feature

 Unsure of confounders:
 - Precipitation affects flow
 - Sunshine affects air temperature
 - Turbidity is affected by flow and precipitation

### Note on lags

If these features are to be used for forecasting, you must make sure that either

1. their influence delay is larger than our forecast horizon (if it's delayed > 4 days, can use data from now to predict 4 days into the future)
1. there is a forecast available for the data (we can use existing/official forecasts for air temperature, precipitation and flow to help forecasts further out)
1. we are also forecasting that variable to be able to use it autoregressively for future forecasts

If none of these are true, we cannot use it to forecast for the desired horizon and need to shorten, drop the feature or forecast it ourselves (bullet 3).

### Note on frequency

For some features, like the precipitation, we probably don't want the average of 10 min totals over an hour (agg mean) but rather the total over an hour (agg sum).
Same goes for sunshine duration for example.

### Note on availability

Not all of those features might be available as far back as the water temperature.
If we find that some feature would be really nice for training, but it's only available for the last few years,
we might need to consider using less data for training and validation. If performance is not good enough, could think about pre-training.

In [ ]:
df = store.query(
    ("2024-01-01", "2025-01-01"),
    [
        "hydro/temperature:mean_1h@bern",
        "hydro/temperature:mean_1h@thun",
        "hydro/flow:mean_1h@bern",
        "hydro/flow:mean_1h@thun",
        "smn/rr:sum_1h@bern",
        "smn/rr:sum_1h@thun",
        "smn/ss:sum_1h@bern",
        "smn/rad:sum_1h@bern",
        # "hydro/turbidity:mean_1h@bern" <- not returned apparently? must investigate
        "smn/tt:mean_1h@bern",
    ],
)
df

In [ ]:
df = resample(df)
df

In [ ]:
df.isna().sum()

In [ ]:
df = interpolate(df, drop_filled=True, columns=None)
df

In [ ]:
df.isna().sum()

In [ ]:
scaler = StandardScaler()
df_with_index = df.set_index(TIME)
scaled_values = scaler.fit_transform(df_with_index.to_numpy(copy=False))
df_s = pd.DataFrame(scaled_values, index=df_with_index.index, columns=df_with_index.columns).reset_index()

In [ ]:
df.describe()

In [ ]:
df_s.describe()

## Temperature Thun -> Temperature Bern

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "temperature_thun"])

In [ ]:
temp_bern = to_ts(df, col="temperature_bern")
temp_thun = to_ts(df, col="temperature_thun")

In [ ]:
plot_ccf(temp_bern, temp_thun, max_lag=4 * 24)

In [ ]:
plot_ccf(remove_seasonality(temp_bern, freq=24), remove_seasonality(temp_thun, freq=24), max_lag=4 * 24)

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, freq=24, method="STL", model=SeasonalityMode.ADDITIVE),
    remove_seasonality(temp_thun, freq=24, method="STL", model=SeasonalityMode.ADDITIVE),
    max_lag=4 * 24,
)

Over a long period, the correlation between temp THUN and BERN is very high everywhere, even if we remove the daily seasonality.
But we can also check if the change in THUN has a correlation with the change in BERN, since that is a stationary series.

In [ ]:
plot_ccf(temp_bern.diff(), temp_thun.diff(), max_lag=4 * 24)

In [ ]:
plot_ccf(remove_seasonality(temp_bern, freq=24).diff(), remove_seasonality(temp_thun, freq=24).diff(), max_lag=4 * 24)

In [ ]:
# switching the places shows that temp_thun barely has any correlation with lagged values of temp_bern, meaning
# temp_thun is leading/ahead of temp_bern (see bern lag 0 is correlated with thun lag 4).
plot_ccf(remove_seasonality(temp_thun, freq=24).diff(), remove_seasonality(temp_bern, freq=24).diff(), max_lag=4 * 24)

And indeed, it seems that there is a peak correlation at around 3-4 hours, meaning when the temperature changes in THUN, it will likely also change in BERN 3-4 hours later.

IMPORTANT EDIT: Technically this means that a positive change in the _rate of change_ of the temperature in THUN will likely lead to a positive change in _rate of change_ of the temperature in BERN 3-4 hours later. I think this can be interpreted as: sudden/quick changes in THUN will lead to sudden/quick changes in BERN a bit later. Given the domain, this can probably be traced back to the more naive interpretation, which would be applicable if we didn't difference the series.

The lagged temperature (change) from THUN might therefore be an interesting feature for forecasting the temperature in BERN.
We can try to reinforce that with Granger Causality, but note that this test is primarily used for economics IIRC.

In [ ]:
# see p-value (second value) is > 0.05 -> likely non-stationary
stationarity_test_adf(temp_bern)

In [ ]:
# see p-value (second value) is << 0.05 -> very likely stationary
stationarity_test_adf(temp_bern.diff())

In [ ]:
# noinspection PyNoneFunctionAssignment
gc_report = granger_causality_tests(temp_thun.diff(), temp_bern.diff(), maxlag=8)

In [ ]:
gc_report

In [ ]:
# Print for all lags, whether the null hypothesis is accepted.
# The null hypothesis is: temp_thun does NOT granger-cause temp_bern, meaning False = there could be causality.
{k: all(s[1] > 0.05 for s in v[0].values()) for k, v in gc_report.items()}

Although this Granger Causality Test clearly rejects the null hypothesis, it's still possible that the alternative hypothesis (temp_thun causes temp_bern) is wrong,
because there is 1) a very high correlation between the two and 2) there are many confounders that influence both temp_thun and temp_bern.
But then again, it's also possible that temp_thun is a useful feature for forecasting temp_bern, even if there were none or barely any causality (if we want to use that word).

Interestingly, the p-values for lag 1 are highest, then lag 2, meaning it's less likely that lag 1 thun is causing bern than lag 3 thun causing bern.
But they are so incredibly low that I don't think we should overthink that.
It does kinda make sense though, because the lowest lags are the furthers away from having any influence, the water simply isn't there yet.

## Flow Bern -> Temperature Bern

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "flow_bern"])

In [ ]:
px.scatter(df_s, x=TIME, y=["temperature_bern", "flow_bern"], title="temp_bern and flow_bern normalized")

In [ ]:
temp_bern_s = to_ts(df_s, col="temperature_bern")
flow_bern = to_ts(df, col="flow_bern")
flow_bern_s = to_ts(df_s, col="flow_bern")

In [ ]:
plot_ccf(temp_bern, flow_bern, max_lag=4 * 24)

In [ ]:
# for ccf, it does not matter whether the data is normalized or not
plot_ccf(temp_bern_s, flow_bern_s, max_lag=4 * 24)

In [ ]:
plot_ccf(temp_bern.diff(), flow_bern.diff(), max_lag=4 * 24)

In [ ]:
plot_ccf(flow_bern.diff(), temp_bern.diff(), max_lag=4 * 24)

We can observe a significant negative correlation at lag 1 for DIFF FLOW -> DIFF TEMPERATURE. Switching the positions shows that flow is leading the temperature because lagging the flow results in higher correlations that lagging the temperature.


~~I would interpret this as: when the flow increases, an hour later the temperature will decrease, which I think might make sense.~~ \
I would interpret this as: when the increase of the flow goes up (high rate of change, e.g. sudden increase), then the rate of change of the temperature _decreases_. This could mean that temperatures either stabilizes or _turns around_ (if it was going up steadily and more water is added, that increase could be slowed because more water needs to be heated).

This means the flow at lag 1 might be an interesting feature to forecast the temperature, but it's also possible that this correlation is already modeled by other relationships with confounding variables (i.e. if precipitation has a negative correlation with temperature and also a positive correlation with the flow, it might not be necessary to include both variables).


In [ ]:
plot_ccf(abs(temp_bern.diff()), abs(flow_bern.diff()), max_lag=4 * 24)

In [ ]:
# FYI: same result whether you remove seasonality before or after diffing
plot_ccf(
    remove_seasonality(temp_bern.diff(), freq=24, model=SeasonalityMode.ADDITIVE), flow_bern.diff(), max_lag=4 * 24
)

Looking at the absolute rates of change and the NCC after trying to remove the daily seasonality still present in the temp diffs, it doesn't confirm or deny anything. Maybe it shows that it's not as significant of a correlation as others.

## Precipitation -> Temperature and Flow?

## Air temperature -> Water temperature

## Sunshine duration -> Water temperature (and air temperature?)

## Global exposure -> Water temperature

But first check corr with sunshine duration and air temp.